# Fine-tune Omnilingual ASR (LoRA) di Colab

Notebook ini fine-tune **`omniASR_LLM_300M_v2`** (Meta omnilingual-asr, 1600+ bahasa) dengan **LoRA** — muat di GPU **T4 gratis** Colab.

Alur: install → load dataset (contoh: Common Voice Indonesia) → LoRA training → evaluasi WER → export & upload ke HuggingFace.

> Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Install dependencies (~3 menit)
!pip install -q omnilingual-asr datasets jiwer soundfile pyyaml

# Toolkit fine-tune — install dari repo GitHub (ganti dengan repo kamu setelah push)
!pip install -q git+https://github.com/emhaihsan/omniasr-toolkit.git

# Atau jika upload folder manual ke Colab:
# import os, sys
# assert os.path.isdir("/content/omniasr_ft"), "Upload folder omniasr_ft/ ke /content/"
# sys.path.insert(0, "/content")

In [ ]:
# 2. Login HuggingFace (Common Voice itu gated — perlu accept license + token)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Konfigurasi fine-tune
from omniasr_ft.train import FinetuneConfig, train

config = FinetuneConfig(
    model_card="omniASR_LLM_300M_v2",
    # Dataset — ganti sesuai kebutuhanmu
    dataset="mozilla-foundation/common_voice_17_0",
    dataset_config="id",          # bahasa Indonesia
    train_split="train",
    valid_split="validation",
    text_column="sentence",
    audio_column="audio",
    lang="ind_Latn",              # kode bahasa omnilingual untuk LID conditioning
    max_train_samples=5000,        # kecilkan untuk eksperimen cepat
    max_valid_samples=200,
    max_secs=20.0,
    # LoRA
    lora_r=16,
    lora_alpha=32,
    # Optimization — batch efektif 16
    batch_size=4,
    grad_accum=4,
    lr=1e-4,
    warmup_steps=100,
    max_steps=1000,
    output_dir="/content/outputs/omniasr_lora_id",
)

In [ ]:
# 4. Training (T4: ~1-2 jam untuk 1000 step; model 6 GB di-download dulu sekali)
model, tokenizer = train(config)

In [ ]:
# 5. Evaluasi WER: base vs fine-tuned
import torch, jiwer
from datasets import load_dataset
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omniasr_ft.infer import load_finetuned_pipeline

test_ds = load_dataset(config.dataset, config.dataset_config, split="test").select(range(50))
audio = [{"waveform": x["audio"]["array"], "sample_rate": x["audio"]["sampling_rate"]} for x in test_ds]
refs = [x[config.text_column] for x in test_ds]
langs = [config.lang] * len(audio)

ft_pipeline = load_finetuned_pipeline(
    config.model_card, f"{config.output_dir}/lora_final.pt",
    lora_r=config.lora_r, lora_alpha=config.lora_alpha)
hyps_ft = ft_pipeline.transcribe(audio, lang=langs, batch_size=4)
del ft_pipeline; torch.cuda.empty_cache()

base_pipeline = ASRInferencePipeline(model_card=config.model_card, dtype=torch.float32)
hyps_base = base_pipeline.transcribe(audio, lang=langs, batch_size=4)
del base_pipeline; torch.cuda.empty_cache()

print(f"WER base      : {jiwer.wer(refs, hyps_base):.3f}")
print(f"WER fine-tuned: {jiwer.wer(refs, hyps_ft):.3f}")
for r, b, f in list(zip(refs, hyps_base, hyps_ft))[:5]:
    print(f"\nREF : {r}\nBASE: {b}\nFT  : {f}")

In [ ]:
# 6. Export: merge LoRA -> checkpoint fairseq2 + asset card, lalu upload ke HF Hub
from omniasr_ft.export import export, push_to_hub

NAME = "omniASR_LLM_300M_v2_id"
HF_REPO = ""  # contoh: "username/omniASR-LLM-300M-id" (kosongkan untuk skip upload)

export(
    model_card=config.model_card,
    adapter_path=f"{config.output_dir}/lora_final.pt",
    name=NAME,
    output_dir="/content/outputs/export",
    lora_r=config.lora_r,
    lora_alpha=config.lora_alpha,
)
if HF_REPO:
    push_to_hub(HF_REPO, "/content/outputs/export", NAME)

## Pakai model hasil fine-tune

```python
from omniasr_ft.infer import load_finetuned_pipeline
pipeline = load_finetuned_pipeline("omniASR_LLM_300M_v2", "lora_final.pt")
print(pipeline.transcribe(["audio.wav"], lang=["ind_Latn"]))
```

Atau dengan checkpoint merged: salin `omniASR_LLM_300M_v2_id.yaml` ke `~/.config/fairseq2/assets/` lalu `ASRInferencePipeline(model_card="omniASR_LLM_300M_v2_id")`.